# Phase 2: NLP Jargon Detection Engine
**Collaborator:** [Person 2 Name]

**Task:** Build the "Detection Brain" using automated dataset scanning. 
We are linking directly to the processed data created by **Person 1** in the `/tmp/MedJargon/` directory.

In [15]:
import pandas as pd
import json
import spacy
import re
import os

# 1. Load NLP model
try:
    nlp = spacy.load("en_core_web_sm")
except:
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# 2. MATCH THE PATH FROM PERSON 1
# This is the "Professional Secret": Person 2 uses Person 1's output folder
repo_path = '/tmp/MedJargon/' 
train_path = os.path.join(repo_path, 'train.csv')
jargon_path = os.path.join(repo_path, 'jargon.json')

# Check if they exist in the temp folder
if os.path.exists(train_path) and os.path.exists(jargon_path):
    df = pd.read_csv(train_path)
    with open(jargon_path, 'r') as f:
        jargon_json = json.load(f)
    print(f"✓ Success: Person 2 is now using the data from {repo_path}")
else:
    # FALLBACK: If the temp folder was cleared, look in the current folder
    if os.path.exists('train.csv'):
        df = pd.read_csv('train.csv')
        with open('jargon.json', 'r') as f:
            jargon_json = json.load(f)
        print("✓ Success: Found files in the local directory.")
    else:
        print("⚠ ERROR: Files not found. Please run the Person 1 notebook first!")

✓ Success: Person 2 is now using the data from /tmp/MedJargon/


In [16]:
def build_expert_dictionary(j_json, train_df):
    dictionary = {}
    
    # 1. Extract terms from jargon.json (specific to your JSON structure)
    for entry in j_json:
        if 'entities' in entry:
            for entity in entry['entities']:
                # Index 3 contains the list of words for the jargon
                term = " ".join(entity[3]).lower()
                dictionary[term] = "Medical Term"

    # 2. Extract definitions from train.csv patterns: "Jargon (Simple)"
    pattern = re.compile(r'([a-zA-Z\s\-]{3,})\s\(([^)]+)\)')
    for text in train_df['target_text'].dropna():
        matches = pattern.findall(text)
        for jargon, simple in matches:
            dictionary[jargon.strip().lower()] = simple.strip().lower()
            
    return dictionary

# Create the global variable for your 'brain'
medical_knowledge = build_expert_dictionary(jargon_json, df)
print(f"✓ Person 2 Task: System learned {len(medical_knowledge)} medical terms.")

✓ Person 2 Task: System learned 6422 medical terms.


In [17]:
def predict_jargon(text):
    doc = nlp(text)
    findings = []
    
    for token in doc:
        # Use Lemmatization (root form) so 'cramping' matches 'cramp'
        root = token.lemma_.lower()
        if root in medical_knowledge:
            findings.append({
                "Found Word": token.text,
                "Simplification": medical_knowledge[root]
            })
            
    return pd.DataFrame(findings)

# Final test for Person 2
sample_text = "The patient presents with severe edema and acute tachycardia."
results = predict_jargon(sample_text)
results

,Found Word,Simplification
0,and,4
1,acute,Medical Term
